# Modeling

In this notebook, the model for identifying which variant of the fictitious website is better (in terms of conversion rate) will be defined and built.

Firstly, some theory is needed to support model development. When analyzing user interaction, there are only two possible actions: **to convert or not**. Being a binary outcome, it can be modeled by a **Bernoulli** probability distribution with **1** indicating success/conversion and **0** indicating failure/no conversion. Also, we have no reason to differentiate any user from the others, so we can assume their probability of converting is the same!

However, the interest isn't on each user itself, but on the global conversion rate for each variant. We can't inspect the behavior of all potential users, so we use a **sample** to try and make inferences about the global conversion rate for each variant. In this project, we generated 1000 observations for each group. Besides that, we have no reason to think that one user's conversion will impact on another user's conversion, so we can assume that the observations are independent of each other.

Considering the number of observations to be fixed, one of the main interests is on the **number of conversions**, given that the **sample conversion rate** can then be obtained by dividing the number of conversions by the number of observations. Respecting the assumptions given above (independence of observations and each having the same probability), we can model the number of conversions as a random variable that follows a **Binomial** probability distribution.

The idea of Bayesian Inference is that we start with a **prior** probability distribution, which is subjective and conditional on the information available at the moment, and then update this probability distribution as new information (our data) becomes available, resulting in a **posterior** probabilty distribution. In more mathematical terms, we have:

$$Posterior \propto Likelihood * Prior$$

in which the **likelihood** is equal to the **joint probability distribution** of observing the observations in the dataset

Since what we want to infer is the conversion rate for each variant, which, by definition, is a value in the interval $(0, 1)$, a reasonable **prior** is a **Beta** probability distribution, with $\alpha = 1, \beta = 1$, in which $\alpha$ and $\beta$ are usually interpreted as **pseudo-counts**. The **likelihood**, as mentioned above, can be modeled by a **Binomial** probability distribution. The great thing about this combination of prior and likelihood is that the **posterior** is also a **Beta** probability distribution, which makes it really easy to update and generate the posterior!

Being more precise, we have:
$$
\alpha_{new} = \alpha_{prior} + conversions

\\

\beta_{new} = \beta_{prior} + not~conversions
$$

Now, let's implement it in code!

## Importing

In [1]:
# Importing stuff
import pandas as pd
import numpy as np

In [2]:
# Reading dataset
conversions_df = pd.read_csv("../data/ab_test_data.csv")

## Variant A modeling

In [3]:
# Defining prior parameters
prior_alpha_a = 1
prior_beta_a = 1

In [4]:
# Calculating number of conversions and no-conversions for variant A
conversions_a = conversions_df[(conversions_df["group"] == "A") & (conversions_df["converted"] == 1)].shape[0]
no_conversions_a = 1000 - conversions_a

In [5]:
# Defining posterior parameters
posterior_alpha_a = prior_alpha_a + conversions_a
posterior_beta_a = prior_beta_a + no_conversions_a

## Variant B modeling

In [6]:
# Defining prior parameters
prior_alpha_b = 1
prior_beta_b = 1

In [8]:
# Calculating number of conversions and no-conversions for variant B
conversions_b = conversions_df[(conversions_df["group"] == "B") & (conversions_df["converted"] == 1)].shape[0]
no_conversions_b = 1000 - conversions_b

In [9]:
# Defining posterior parameters
posterior_alpha_b = prior_alpha_b + conversions_b
posterior_beta_b = prior_beta_b + no_conversions_b

## Monte Carlo simulations

Now that the posterior distribution for each variant was obtained, the goal is to compare them to identify which variant has the higher conversion rate. One option would be to calculate some integrals and do some more advanced calculations, but this would difficult stakeholders' understanding. The approach chosen consists of Monte Carlo simulations for various random samples drawn from the posterior probability distribution for each variant and compare those simulations!

In [10]:
# Generating conversions rates from the posterior probability distributions of each variant
generated_conv_rates_a = np.random.beta(posterior_alpha_a, posterior_beta_b, size=100000)
generated_conv_rates_b = np.random.beta(posterior_alpha_b, posterior_beta_b, size=100000)

In [11]:
# Calculating the probability of variant B having a higher conversion rate than variant A
prob_b_better = np.mean(generated_conv_rates_b > generated_conv_rates_a)
prob_b_better

np.float64(0.94055)

Since the probability of variant B having a higher conversion rate than variant A, the choice, based on the calculated probability, would be to **follow on with variant B** of the fictional website!

## Expected Loss

Having the probability of one variant being better than other is great, but, oftentimes, business stakeholders are interested in the risk associated with the decision making: If I choose variant B, but, actually, variant A has the highest conversion rate, how much do I "lose"?

To make the decision making process robust, let's define a threshold of caring: if the expected loss in conversion rate is less than 0,2%, the decision is validated by the criteria of expected loss

In [13]:
# Calculating the expected loss in conversion rate of choosing B when A is the better choice
loss_choosing_b = np.maximum(generated_conv_rates_a - generated_conv_rates_b, 0)
expc_loss_choosing_b = np.mean(loss_choosing_b)
expc_loss_choosing_b

np.float64(0.0003951888403122464)

In [14]:
# Calculating the expected loss in conversion rate of choosing A when B is the better choice
# (just for sanity check)
loss_choosing_a = np.maximum(generated_conv_rates_b - generated_conv_rates_a, 0)
expc_loss_choosing_a = np.mean(loss_choosing_a)
expc_loss_choosing_a

np.float64(0.02465160042443459)

Since the smaller expected loss is the one when choosing B when A is the better choice and the expected loss is less than 0,2%, than our choice of the B variant is validated!